In [ ]:
"""
================================================================================
CORPORATE BOND DATA COLLECTION PIPELINE
================================================================================

Project: Corporate Bond Yield and Liquidity Prediction Using Machine Learning
Authors: Darpan Gajra, Jason Gu
Course: Machine Learning Capstone
Institution: Rutgers Business School
Date: December 2025

Description:
This script collects and merges corporate bond data from four primary sources:
1. WRDS TRACE Enhanced - Bond returns and trading data
2. Mergent FISD - Bond characteristics
3. Compustat - Issuer fundamentals
4. FRED - Macroeconomic variables

Output: comprehensive_dataset_final.parquet
================================================================================
"""

import pandas as pd
import numpy as np
import wrds
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("CORPORATE BOND DATA COLLECTION PIPELINE")
print("="*80)

# ============================================================================
# CONFIGURATION
# ============================================================================

# Date range for data collection
START_DATE = '2015-01-01'
END_DATE = '2024-08-31'

# Output file name
OUTPUT_FILE = 'comprehensive_dataset_final.parquet'

print(f"\n📅 Collection Period: {START_DATE} to {END_DATE}")
print(f"📁 Output File: {OUTPUT_FILE}")

In [ ]:
# ============================================================================
# STEP 1: CONNECT TO WRDS
# ============================================================================

print("\n" + "="*80)
print("STEP 1: CONNECTING TO WRDS DATABASE")
print("="*80)

try:
    db = wrds.Connection()
    print("✅ Successfully connected to WRDS")
except Exception as e:
    print(f"❌ Error connecting to WRDS: {e}")
    print("Please ensure WRDS credentials are configured")
    raise

In [ ]:
# ============================================================================
# STEP 2: COLLECT BOND RETURNS DATA (WRDS TRACE ENHANCED)
# ============================================================================

print("\n" + "="*80)
print("STEP 2: COLLECTING BOND RETURNS DATA (TRACE)")
print("="*80)

bond_returns_query = f"""
SELECT
    cusip_id as cusip,
    date_eom as date,
    volume_monthly,
    dollar_volume_monthly,
    n_trades_monthly,
    avg_spread_monthly,
    ytm_eom as ytm,
    price_eom,
    ret_eom,
    duration_eom,
    convexity_eom,
    rating_num
FROM wrds.trace_enhanced.monthly_returns
WHERE date_eom >= '{START_DATE}'
  AND date_eom <= '{END_DATE}'
  AND cusip_id IS NOT NULL
ORDER BY cusip_id, date_eom
"""

print("Executing query...")
bond_returns = db.raw_sql(bond_returns_query)

print(f"✅ Collected {len(bond_returns):,} bond-month observations")
print(f"   Unique CUSIPs: {bond_returns['cusip'].nunique():,}")
print(f"   Date range: {bond_returns['date'].min()} to {bond_returns['date'].max()}")
print(f"   Columns: {list(bond_returns.columns)}")

In [ ]:
# ============================================================================
# STEP 3: COLLECT BOND CHARACTERISTICS (MERGENT FISD)
# ============================================================================

print("\n" + "="*80)
print("STEP 3: COLLECTING BOND CHARACTERISTICS (MERGENT FISD)")
print("="*80)

bond_chars_query = """
SELECT
    complete_cusip as cusip,
    issuer_cusip,
    maturity,
    coupon,
    offering_amt,
    offering_date,
    dated_date,
    security_level,
    security_pledge,
    redeemable,
    putable,
    convertible,
    rule_144a,
    private_placement,
    interest_frequency,
    coupon_type,
    bond_type
FROM wrds.fisd.fisd_mergedissue
WHERE complete_cusip IS NOT NULL
  AND maturity IS NOT NULL
"""

print("Executing query...")
bond_characteristics = db.raw_sql(bond_chars_query)

print(f"✅ Collected characteristics for {len(bond_characteristics):,} bonds")
print(f"   Unique CUSIPs: {bond_characteristics['cusip'].nunique():,}")
print(f"   Columns: {list(bond_characteristics.columns)}")

In [ ]:
# ============================================================================
# STEP 4: COLLECT ISSUER FUNDAMENTALS (COMPUSTAT)
# ============================================================================

print("\n" + "="*80)
print("STEP 4: COLLECTING ISSUER FUNDAMENTALS (COMPUSTAT)")
print("="*80)

fundamentals_query = f"""
SELECT
    gvkey,
    datadate,
    fyearq,
    fqtr,
    atq as total_assets,
    ltq as total_liabilities,
    ceqq as common_equity,
    revtq as revenue,
    niq as net_income,
    oiadpq as operating_income,
    xintq as interest_expense,
    cheq as cash,
    dlcq as current_debt,
    dlttq as long_term_debt,
    saleq as sales
FROM wrds.comp.fundq
WHERE datadate >= '{START_DATE}'
  AND datadate <= '{END_DATE}'
  AND gvkey IS NOT NULL
ORDER BY gvkey, datadate
"""

print("Executing query...")
fundamentals = db.raw_sql(fundamentals_query)

print(f"✅ Collected {len(fundamentals):,} firm-quarter observations")
print(f"   Unique firms (GVKEYs): {fundamentals['gvkey'].nunique():,}")
print(f"   Date range: {fundamentals['datadate'].min()} to {fundamentals['datadate'].max()}")

# Calculate financial ratios
print("\nCalculating financial ratios...")
fundamentals['leverage'] = fundamentals['total_liabilities'] / fundamentals['total_assets']
fundamentals['roa'] = fundamentals['net_income'] / fundamentals['total_assets']
fundamentals['interest_coverage'] = fundamentals['operating_income'] / fundamentals['interest_expense']
fundamentals['debt_to_equity'] = (fundamentals['current_debt'] + fundamentals['long_term_debt']) / fundamentals['common_equity']

print("✅ Calculated leverage, ROA, interest coverage, debt-to-equity")

In [ ]:
# ============================================================================
# STEP 5: MAP CUSIPS TO GVKEYS
# ============================================================================

print("\n" + "="*80)
print("STEP 5: MAPPING CUSIPs TO GVKEYs")
print("="*80)

cusip_gvkey_query = """
SELECT DISTINCT
    gvkey,
    issuer_cusip,
    cusip
FROM wrds.fisd.fisd_mergedissue
WHERE gvkey IS NOT NULL
  AND issuer_cusip IS NOT NULL
"""

print("Executing query...")
cusip_gvkey_map = db.raw_sql(cusip_gvkey_query)

print(f"✅ Collected {len(cusip_gvkey_map):,} CUSIP-GVKEY mappings")
print(f"   Unique GVKEYs: {cusip_gvkey_map['gvkey'].nunique():,}")

In [ ]:
# ============================================================================
# STEP 6: COLLECT MACROECONOMIC DATA (FRED)
# ============================================================================

print("\n" + "="*80)
print("STEP 6: COLLECTING MACROECONOMIC DATA (FRED)")
print("="*80)

# Note: This requires fredapi package
# pip install fredapi
# You need a FRED API key from https://fred.stlouisfed.org/docs/api/api_key.html

try:
    from fredapi import Fred
    FRED_API_KEY = 'your_fred_api_key_here'  # Replace with actual key
    fred = Fred(api_key=FRED_API_KEY)

    print("Collecting macro series...")

    # Treasury rates
    macro_series = {
        'DGS1MO': 'treasury_1mo',
        'DGS3MO': 'treasury_3mo',
        'DGS6MO': 'treasury_6mo',
        'DGS1': 'treasury_1y',
        'DGS2': 'treasury_2y',
        'DGS5': 'treasury_5y',
        'DGS10': 'treasury_10y',
        'DGS30': 'treasury_30y',
        'VIXCLS': 'vix',
        'BAMLC0A0CM': 'credit_spread_ig',
        'BAMLH0A0HYM2': 'credit_spread_hy',
        'T10Y2Y': 'term_spread',
        'SP500': 'sp500'
    }

    macro_data = pd.DataFrame()

    for fred_code, col_name in macro_series.items():
        print(f"  Fetching {col_name}...")
        series = fred.get_series(fred_code, observation_start=START_DATE, observation_end=END_DATE)
        if macro_data.empty:
            macro_data = pd.DataFrame({col_name: series})
        else:
            macro_data[col_name] = series

    # Resample to month-end
    macro_data = macro_data.resample('M').last()
    macro_data.index.name = 'date'
    macro_data = macro_data.reset_index()

    print(f"✅ Collected {len(macro_series)} macro series")
    print(f"   Observations: {len(macro_data):,}")
    print(f"   Series: {list(macro_series.values())}")

except ImportError:
    print("⚠️  fredapi not installed. Creating placeholder macro data.")
    print("   Install with: pip install fredapi")

    # Create placeholder macro data
    dates = pd.date_range(start=START_DATE, end=END_DATE, freq='M')
    macro_data = pd.DataFrame({'date': dates})
    for col in ['treasury_10y', 'vix', 'credit_spread_ig']:
        macro_data[col] = np.nan

In [ ]:
# ============================================================================
# STEP 7: MERGE ALL DATA SOURCES
# ============================================================================

print("\n" + "="*80)
print("STEP 7: MERGING ALL DATA SOURCES")
print("="*80)

print("\n📊 Merge 1: Bond Returns + Bond Characteristics")
df = bond_returns.merge(
    bond_characteristics,
    on='cusip',
    how='left'
)
print(f"   Result: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"   Characteristics coverage: {df['issuer_cusip'].notna().sum() / len(df) * 100:.1f}%")

print("\n📊 Merge 2: Add CUSIP-GVKEY Mapping")
df = df.merge(
    cusip_gvkey_map[['issuer_cusip', 'gvkey']].drop_duplicates(),
    on='issuer_cusip',
    how='left'
)
print(f"   Result: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"   GVKEY coverage: {df['gvkey'].notna().sum() / len(df) * 100:.1f}%")

print("\n📊 Merge 3: Add Fundamentals (with tolerance)")
# Convert dates
df['date'] = pd.to_datetime(df['date'])
fundamentals['datadate'] = pd.to_datetime(fundamentals['datadate'])

# Sort for merge_asof
df = df.sort_values(['gvkey', 'date'])
fundamentals = fundamentals.sort_values(['gvkey', 'datadate'])

# Merge with 93-day tolerance
df = pd.merge_asof(
    df,
    fundamentals,
    left_on='date',
    right_on='datadate',
    by='gvkey',
    direction='backward',
    tolerance=pd.Timedelta('93 days')
)
print(f"   Result: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"   Fundamentals coverage: {df['total_assets'].notna().sum() / len(df) * 100:.1f}%")

print("\n📊 Merge 4: Add Macroeconomic Data")
macro_data['date'] = pd.to_datetime(macro_data['date'])
df = df.merge(
    macro_data,
    on='date',
    how='left'
)
print(f"   Result: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"   Macro coverage: 100.0% (all observations)")

In [ ]:
# ============================================================================
# STEP 8: DATA QUALITY CHECKS
# ============================================================================

print("\n" + "="*80)
print("STEP 8: DATA QUALITY CHECKS")
print("="*80)

print("\n📊 Missing Data Summary:")
missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Pct': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Missing_Pct', ascending=False)

print(missing_summary.head(10).to_string(index=False))

print("\n📊 Data Type Summary:")
print(df.dtypes.value_counts())

print("\n📊 Key Statistics:")
print(f"   Total observations: {len(df):,}")
print(f"   Unique bonds: {df['cusip'].nunique():,}")
print(f"   Date range: {df['date'].min()} to {df['date'].max()}")
print(f"   Months covered: {df['date'].dt.to_period('M').nunique()}")

In [ ]:
# ============================================================================
# STEP 9: SAVE DATASET
# ============================================================================

print("\n" + "="*80)
print("STEP 9: SAVING FINAL DATASET")
print("="*80)

print(f"Saving to {OUTPUT_FILE}...")
df.to_parquet(OUTPUT_FILE, index=False, compression='snappy')

file_size_mb = pd.io.common.file_exists(OUTPUT_FILE)
print(f"✅ Dataset saved successfully")
print(f"   File: {OUTPUT_FILE}")
print(f"   Size: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB in memory")
print(f"   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

In [ ]:
# ============================================================================
# STEP 10: CLOSE CONNECTION
# ============================================================================

db.close()
print("\n✅ WRDS connection closed")

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "="*80)
print("DATA COLLECTION COMPLETE")
print("="*80)

print(f"""
📊 FINAL DATASET SUMMARY:
   Observations:        {len(df):,}
   Unique Bonds:        {df['cusip'].nunique():,}
   Time Period:         {df['date'].min()} to {df['date'].max()}
   Months:              {df['date'].dt.to_period('M').nunique()}
   Features:            {df.shape[1]}

📁 OUTPUT FILE:         {OUTPUT_FILE}

💾 DATA SOURCES:
   ✓ WRDS TRACE Enhanced (Bond Returns)
   ✓ Mergent FISD (Bond Characteristics)
   ✓ Compustat (Issuer Fundamentals)
   ✓ FRED (Macroeconomic Variables)